# Day 10 — Segment Interpretation & Business Profiling

This notebook translates the K-Means cluster IDs from Day 9 into data-driven, business-facing customer archetypes. It does not change the clustering model.

**Interpretation rules**
- Profile each cluster with customer count and RFM mean/median.
- Compare cluster-level mean Recency, Frequency and Monetary values against the median cluster profile.
- Lower Recency is better because it means a more recent purchase. Higher Frequency and Monetary are better for engagement/value.
- Assign descriptive labels from those relative comparisons. Labels are archetypes, not ground-truth classes or predictions.
- Translate each archetype into retention, reactivation, cross-sell or value-development actions. Actual cluster sizes and metrics are generated at runtime; nothing is hard-coded.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.eda import positive_sales_view
from src.rfm_analysis import build_rfm
from src.clustering_prep import prepare_clustering_features
from src.kmeans_clustering import evaluate_k_range, fit_kmeans
from src.segment_interpretation import profile_clusters, add_relative_segment_labels

online_retail = fetch_ucirepo(id=352)
raw = online_retail.data.features.copy()
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['invoice_date'] = pd.to_datetime(raw['invoice_date'], errors='coerce')
for col in ['quantity', 'unit_price', 'customer_id']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
clean = raw.drop_duplicates().copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
sales = positive_sales_view(clean)
rfm = build_rfm(sales)
_, scaled_features, _ = prepare_clustering_features(rfm)
evaluation = evaluate_k_range(scaled_features, range(2, 9), random_state=42, n_init=20)
selected_k = int(evaluation.loc[evaluation['silhouette_score'].idxmax(), 'k'])
model, labels = fit_kmeans(scaled_features, selected_k, random_state=42, n_init=20)
clustered_rfm = rfm.loc[scaled_features.index].copy()
clustered_rfm['cluster'] = labels
print(f'Selected k from Day 9 methodology: {selected_k}')
print(f'Customers profiled: {len(clustered_rfm):,}')

## 1. Cluster profiles

In [ ]:
profile = profile_clusters(clustered_rfm)
profile

## 2. Relative business labels

The labels below use the median of the observed cluster means as the comparison point. This avoids inventing absolute spend or engagement cutoffs. Because cluster IDs are arbitrary, the business label—not the numeric ID—should be used in recommendations.

In [ ]:
labeled_profile = add_relative_segment_labels(profile)
labeled_profile

In [ ]:
display_columns = ['cluster', 'segment_label', 'customers', 'recency_mean', 'frequency_mean', 'monetary_mean']
labeled_profile[display_columns].sort_values('monetary_mean', ascending=False)

## 3. Business action framework

Actions are intentionally tied to observed RFM characteristics rather than invented campaign performance.

- **High-Value Loyal:** protect retention with loyalty benefits, early access and personalized offers.
- **Recent High-Value:** encourage a second/next purchase and deepen the relationship through relevant cross-sell.
- **Frequent High-Value:** protect repeat behavior and monitor for recency deterioration.
- **Recent Frequent:** increase basket value through bundles and cross-sell.
- **Recent / Developing:** nurture toward repeat purchasing with onboarding and targeted recommendations.
- **Frequent / Lower Value:** focus on basket-size or margin improvement without sacrificing frequency.
- **High-Value / At Risk:** prioritize win-back and personalized reactivation.
- **Lower Engagement:** use lower-cost reactivation tests and avoid overspending on incentives.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for _, row in labeled_profile.iterrows():
    ax.scatter(row['frequency_mean'], row['monetary_mean'], s=max(row['customers'], 20), alpha=0.7)
    ax.annotate(str(row['segment_label']), (row['frequency_mean'], row['monetary_mean']), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel('Cluster mean frequency')
ax.set_ylabel('Cluster mean monetary value')
ax.set_title('Business segment profile: Frequency vs Monetary')
plt.tight_layout()
plt.show()

## Interpretation guardrails

These segments describe the observed customer base and support prioritization; they do not prove causality, future behavior or campaign lift. Before operational use, validate the segment definitions against campaign history, margins and retention outcomes when those data become available.